In [5]:
import os
import re
import time
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict # Import defaultdict

# -----------------------------
# 0) 설정
# -----------------------------
SRC_DIR = r"/content/drive/MyDrive/양상추 분류모델/practice code/images"

DATE_START = "20260209"
DATE_END   = "20260223"

DST_RGB_TOP = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/260209-260223"

DST_RGB_FRONT = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/260209-260223"

DST_THERMAL = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/열화상/0. 원본/260209-260223"

DST_JSON = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/JSON/260209-260223"

# 병렬 스레드 수: 드라이브 I/O면 8~16 정도가 보통 적당
MAX_WORKERS = 12

# 파일명에서 날짜 8자리 찾기 (bedXX_YYYYMMDD_HHMMSS_...)
DATE_RE = re.compile(r"_([0-9]{8})_")

def ensure_dirs():
    for d in [DST_RGB_TOP, DST_RGB_FRONT, DST_THERMAL, DST_JSON]:
        os.makedirs(d, exist_ok=True)

def classify_dst(filename: str):
    """
    규칙:
    - thermal 포함 -> 열화상 (jpg) / json이면 json으로
    - cam0/cam1 -> RGB 윗면 (jpg)
    - cam2 -> RGB 정면 (jpg)
    - .json -> JSON
    """
    lower = filename.lower()

    is_json = lower.endswith(".json")
    is_jpg  = lower.endswith(".jpg") or lower.endswith(".jpeg")

    if not (is_json or is_jpg):
        return None, None  # 무시

    if is_json:
        return DST_JSON, "JSON"

    # 여기부터 jpg류
    if "thermal" in lower:
        return DST_THERMAL, "THERMAL"

    # RGB 분류
    if "cam0" in lower or "cam1" in lower:
        return DST_RGB_TOP, "RGB_TOP"
    if "cam2" in lower:
        return DST_RGB_FRONT, "RGB_FRONT"

    # 그 외 cam 번호면 일단 None (원하면 RGB_TOP으로 보내게 바꿀 수 있음)
    return None, None

def in_date_range(filename: str):
    m = DATE_RE.search(filename)
    if not m:
        return False
    ymd = m.group(1)
    return (DATE_START <= ymd <= DATE_END)

def copy_one(src_path, dst_path, category_name):
    # 재시작 가능: 이미 있으면 스킵
    if os.path.exists(dst_path):
        return "skipped", category_name
    shutil.copy2(src_path, dst_path)
    return "copied", category_name

# -----------------------------
# 1) 대상 파일 리스트 만들기
# -----------------------------
ensure_dirs()

all_files = [] # Initialize all_files to prevent NameError if os.listdir fails
try:
    all_files = os.listdir(SRC_DIR)
except OSError as e:
    print(f"Error listing source directory: {e}")
    print(f"Please check if the path '{SRC_DIR}' is correct and accessible.")
    # Removed exit() to allow graceful continuation in notebook environments
    # If the directory is truly inaccessible, all_files remains empty, and no files will be processed.

targets = []
initial_category_counts = defaultdict(int) # Initialize defaultdict

for fn in all_files:
    if not in_date_range(fn):
        continue
    dst_dir, category_name = classify_dst(fn)
    if dst_dir is None or category_name is None:
        continue
    targets.append((fn, dst_dir, category_name)) # Store category_name as well
    initial_category_counts[category_name] += 1 # Increment count for the category

print(f"전체 파일: {len(all_files):,}")
print(f"선별 대상(날짜+규칙 통과): {len(targets):,}")

# Print initial file counts by category
print("Initial file counts by category:")
for category, count in sorted(initial_category_counts.items()):
    print(f"  - {category}: {count:,}")

# -----------------------------
# 2) 병렬 복사 + 진행률/ETA
# -----------------------------
t0 = time.time()
done = 0
copied = 0
skipped = 0
failed = 0

# New: Track per-category copy status
final_category_counts = defaultdict(lambda: defaultdict(int))

def fmt_eta(sec):
    if sec < 0: sec = 0
    m, s = divmod(int(sec), 60)
    h, m = divmod(m, 60)
    if h: return f"{h}h {m}m {s}s"
    if m: return f"{m}m {s}s"
    return f"{s}s"

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = []
    future_to_category = {} # Map future to its category for error handling
    for fn, dst_dir, category_name in targets: # Unpack category_name
        src_path = os.path.join(SRC_DIR, fn)
        dst_path = os.path.join(dst_dir, fn)
        future = ex.submit(copy_one, src_path, dst_path, category_name) # Pass category_name
        futures.append(future)
        future_to_category[future] = category_name

    total = len(futures)
    for i, fut in enumerate(as_completed(futures), 1):
        done += 1
        category = future_to_category[fut] # Get the category associated with this future
        try:
            status, returned_category = fut.result() # Unpack status and category
            final_category_counts[category][status] += 1 # Update per-category counts
            if status == "copied":
                copied += 1
            else: # status == "skipped"
                skipped += 1
        except Exception as e:
            failed += 1
            final_category_counts[category]["failed"] += 1 # Update per-category failed counts
            # print(f"Error processing {future_to_category[fut]} file (category: {category}): {e}") # Optional: detailed error logging

        # 200개마다 한 번씩 출력(너무 스팸 안 나가게)
        if done % 200 == 0 or done == total:
            elapsed = time.time() - t0
            rate = done / elapsed if elapsed > 0 else 0
            eta = (total - done) / rate if rate > 0 else 0
            print(f"[{done:,}/{total:,}] copied={copied:,} skipped={skipped:,} failed={failed:,} | {rate:.2f} files/s | ETA {fmt_eta(eta)}")

print("완료!")
print(f"copied={copied:,}, skipped={skipped:,}, failed={failed:,}, elapsed={fmt_eta(time.time()-t0)}")

# Print final per-category results
print("\nFinal copy status by category:")
for category, counts in sorted(final_category_counts.items()):
    copied_cat = counts.get("copied", 0)
    skipped_cat = counts.get("skipped", 0)
    failed_cat = counts.get("failed", 0)
    total_cat_processed = copied_cat + skipped_cat + failed_cat
    print(f"  - {category}: Copied={copied_cat:,}, Skipped={skipped_cat:,}, Failed={failed_cat:,}, Total Processed={total_cat_processed:,}")


전체 파일: 252,681
선별 대상(날짜+규칙 통과): 29,904
Initial file counts by category:
  - JSON: 15,083
  - RGB_FRONT: 4,918
  - RGB_TOP: 9,641
  - THERMAL: 262
[200/29,904] copied=0 skipped=200 failed=0 | 31.61 files/s | ETA 15m 39s
[400/29,904] copied=0 skipped=400 failed=0 | 62.79 files/s | ETA 7m 49s
[600/29,904] copied=0 skipped=600 failed=0 | 93.51 files/s | ETA 5m 13s
[800/29,904] copied=0 skipped=800 failed=0 | 123.88 files/s | ETA 3m 54s
[1,000/29,904] copied=178 skipped=822 failed=0 | 51.38 files/s | ETA 9m 22s
[1,200/29,904] copied=378 skipped=822 failed=0 | 34.38 files/s | ETA 13m 54s
[1,400/29,904] copied=578 skipped=822 failed=0 | 27.82 files/s | ETA 17m 4s
[1,600/29,904] copied=778 skipped=822 failed=0 | 24.43 files/s | ETA 19m 18s
[1,800/29,904] copied=978 skipped=822 failed=0 | 22.68 files/s | ETA 20m 39s
[2,000/29,904] copied=1,178 skipped=822 failed=0 | 21.29 files/s | ETA 21m 50s
[2,200/29,904] copied=1,378 skipped=822 failed=0 | 20.43 files/s | ETA 22m 35s
[2,400/29,904] copied=1

# 폴더하부로 넣고 검은사진 삭제하기

In [7]:
import os
import re
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

from PIL import Image
import numpy as np

# =========================
# 0) 네가 이미 쓰던 DST 경로들 넣기
# =========================
DST_RGB_TOP = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/260209-260223"
DST_RGB_FRONT = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/260209-260223"
DST_THERMAL = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/열화상/0. 원본/260209-260223"
DST_JSON = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/JSON/260209-260223"

DATE_RE = re.compile(r"_([0-9]{8})_")  # _YYYYMMDD_

# =========================
# 1) 날짜별 폴더 생성 + 파일 이동
#    (RGB TOP/FRONT는 검은 사진 삭제 후 이동)
# =========================

# "아예 검은 사진" 판정 기준 (필요하면 조절)
BLACK_MEAN_THRESH = 2.0          # 평균 밝기(0~255) 이 값보다 작으면 거의 검정
BLACK_MAX_THRESH  = 10           # 최대 밝기까지도 낮으면 더 확실
DOWNSAMPLE = (128, 128)          # 빠른 판정을 위한 축소 크기

MAX_WORKERS = 12

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def ymd_to_folder(ymd: str) -> str:
    # 20260209 -> 260209
    return ymd[2:]

def extract_ymd(name: str):
    m = DATE_RE.search(name)
    return m.group(1) if m else None

def is_black_image(img_path: Path) -> bool:
    # RGB TOP/FRONT jpg만 들어온다고 가정
    try:
        with Image.open(img_path) as im:
            im = im.convert("RGB").resize(DOWNSAMPLE)
            arr = np.asarray(im, dtype=np.uint8)
            mean_v = float(arr.mean())
            max_v  = float(arr.max())
            return (mean_v < BLACK_MEAN_THRESH) and (max_v < BLACK_MAX_THRESH)
    except Exception:
        # 열기 실패 등은 "검정"으로 취급하지 않고 그냥 넘김(삭제 방지)
        return False

def move_file(src: Path, dst_dir: Path):
    ensure_dir(dst_dir)
    dst = dst_dir / src.name
    if dst.exists():
        return "skipped"
    src.rename(dst)  # 같은 드라이브 내면 rename이 가장 빠름(=move)
    return "moved"

def delete_file(p: Path):
    try:
        p.unlink()
        return "deleted"
    except Exception:
        return "delete_failed"

def process_one_file(root_dir: Path, f: Path, do_black_check: bool):
    """
    root_dir: DST_* 중 하나
    f: root_dir 아래의 파일(하위 폴더 제외하고 파일만 대상으로 잡는 걸 권장)
    do_black_check: RGB TOP/FRONT만 True
    """
    ymd = extract_ymd(f.name)
    if not ymd:
        return ("no_date", str(f))

    # 1) 검은사진 삭제 (RGB TOP/FRONT만)
    if do_black_check and f.suffix.lower() in [".jpg", ".jpeg"]:
        if is_black_image(f):
            r = delete_file(f)
            return (r, str(f))

    # 2) 날짜 폴더로 이동
    folder = ymd_to_folder(ymd)  # 260209
    dst_dir = root_dir / folder
    r = move_file(f, dst_dir)
    return (r, str(f))

def list_files_flat(root_dir: Path):
    # 이미 날짜 폴더가 생긴 뒤 재실행해도 안전하게: "root 바로 아래 파일"만 처리
    return [p for p in root_dir.iterdir() if p.is_file()]

def run_for_root(root, do_black_check=False):
    root_dir = Path(root)
    ensure_dir(root_dir)

    files = list_files_flat(root_dir)
    total = len(files)
    print(f"\n[{root_dir}] files(at root level) = {total:,}")

    stats = {
        "moved": 0, "skipped": 0,
        "deleted": 0, "delete_failed": 0,
        "no_date": 0, "other": 0
    }

    t0 = time.time()
    done = 0

    def fmt_eta(sec):
        sec = max(0, int(sec))
        m, s = divmod(sec, 60)
        h, m = divmod(m, 60)
        if h: return f"{h}h {m}m {s}s"
        if m: return f"{m}m {s}s"
        return f"{s}s"

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(process_one_file, root_dir, f, do_black_check) for f in files]
        for fut in as_completed(futures):
            done += 1
            r, _ = fut.result()
            if r in stats:
                stats[r] += 1
            else:
                stats["other"] += 1

            if done % 200 == 0 or done == total:
                elapsed = time.time() - t0
                rate = done / elapsed if elapsed > 0 else 0
                eta = (total - done) / rate if rate > 0 else 0
                print(f"  [{done:,}/{total:,}] {rate:.2f} files/s | ETA {fmt_eta(eta)} | "
                      f"moved={stats['moved']:,} deleted={stats['deleted']:,} skipped={stats['skipped']:,} no_date={stats['no_date']:,}")

    print("  DONE:", stats, "| elapsed:", fmt_eta(time.time() - t0))

# =========================
# 실행
# =========================

# 1) RGB TOP: 날짜폴더 분류 + 검은사진 삭제
run_for_root(DST_RGB_TOP, do_black_check=True)

# 2) RGB FRONT: 날짜폴더 분류 + 검은사진 삭제
run_for_root(DST_RGB_FRONT, do_black_check=True)

# 3) THERMAL: 날짜폴더 분류만 (삭제X)
run_for_root(DST_THERMAL, do_black_check=False)

# 4) JSON: 날짜폴더 분류만 (삭제X)
run_for_root(DST_JSON, do_black_check=False)

print("\nALL DONE.")


[/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/260209-260223] files(at root level) = 0
  DONE: {'moved': 0, 'skipped': 0, 'deleted': 0, 'delete_failed': 0, 'no_date': 0, 'other': 0} | elapsed: 0s

[/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/260209-260223] files(at root level) = 0
  DONE: {'moved': 0, 'skipped': 0, 'deleted': 0, 'delete_failed': 0, 'no_date': 0, 'other': 0} | elapsed: 0s

[/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/열화상/0. 원본/260209-260223] files(at root level) = 0
  DONE: {'moved': 0, 'skipped': 0, 'deleted': 0, 'delete_failed': 0, 'no_date': 0, 'other': 0} | elapsed: 0s

[/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/JSON/260209-260223] files(at root level) = 0
  DONE: {'moved': 0, 'skipped': 0, 'deleted

In [8]:
import os

def count_files_in_directory(path):
    count = 0
    if os.path.exists(path) and os.path.isdir(path):
        for root, dirs, files in os.walk(path):
            count += len(files)
    return count

dirs_to_count = {
    "DST_RGB_TOP": DST_RGB_TOP,
    "DST_RGB_FRONT": DST_RGB_FRONT,
    "DST_THERMAL": DST_THERMAL,
    "DST_JSON": DST_JSON
}

print("File counts in destination directories:")
for name, path in dirs_to_count.items():
    file_count = count_files_in_directory(path)
    print(f"  - {name} ({path}): {file_count:,} files")


File counts in destination directories:
  - DST_RGB_TOP (/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/260209-260223): 7,867 files
  - DST_RGB_FRONT (/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/260209-260223): 4,918 files
  - DST_THERMAL (/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/열화상/0. 원본/260209-260223): 262 files
  - DST_JSON (/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/JSON/260209-260223): 15,083 files


#검은그림이 안사라져서 한번더 돌리기로 함

In [9]:
import os, re, shutil, time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from PIL import Image
from tqdm import tqdm

# ========= 너가 이미 만들어둔 4개 경로(여기만 맞춰) =========
DST_RGB_TOP = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/260209-260223"
DST_RGB_FRONT = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/260209-260223"
DST_THERMAL = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/열화상/0. 원본/260209-260223"
DST_JSON = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/JSON/260209-260223"

# ==========================================================

DATE_RE = re.compile(r"_([0-9]{8})_")

# ---- 블랙 판정 파라미터(여기 조절) ----
DOWNSAMPLE = (128, 128)

BLACK_MEAN_THRESH = 8.0     # (0~255) 평균이 이 값보다 낮으면 후보
BLACK_P99_THRESH  = 20.0    # p99가 이 값보다 낮으면 거의 검정
BRIGHT_THR        = 30      # "밝은 픽셀" 기준
BRIGHT_RATIO_MAX  = 0.01    # 밝은 픽셀 비율 1% 미만이면 거의 검정
# -------------------------------------

def ensure_date_folders(base_dir, yymmdd_list):
    for d in yymmdd_list:
        os.makedirs(os.path.join(base_dir, d), exist_ok=True)

def extract_yymmdd_from_name(fname):
    m = DATE_RE.search(fname)
    if not m:
        return None
    yyyymmdd = m.group(1)          # 20260210
    return yyyymmdd[2:]            # 260210

def is_black_image(img_path):
    try:
        img = Image.open(img_path).convert("RGB").resize(DOWNSAMPLE)
        arr = np.asarray(img, dtype=np.float32)
        gray = 0.299*arr[:,:,0] + 0.587*arr[:,:,1] + 0.114*arr[:,:,2]

        mean = float(gray.mean())
        p99  = float(np.quantile(gray, 0.99))
        bright_ratio = float((gray > BRIGHT_THR).mean())

        # 블랙 판정: "대부분 검정" + "작은 불빛은 허용"
        if (mean < BLACK_MEAN_THRESH) and (p99 < BLACK_P99_THRESH) and (bright_ratio < BRIGHT_RATIO_MAX):
            return True, (mean, p99, bright_ratio)
        return False, (mean, p99, bright_ratio)

    except Exception as e:
        # 읽기 실패는 삭제 대신 스킵(안전)
        return False, (None, None, None)

def list_files_recursive(root, exts=None):
    out = []
    for dp, dn, fn in os.walk(root):
        for f in fn:
            if exts is None:
                out.append(os.path.join(dp, f))
            else:
                lf = f.lower()
                if any(lf.endswith(x) for x in exts):
                    out.append(os.path.join(dp, f))
    return out

def move_into_date_folder(path, base_dir):
    fname = os.path.basename(path)
    yymmdd = extract_yymmdd_from_name(fname)
    if yymmdd is None:
        return False
    dst_dir = os.path.join(base_dir, yymmdd)
    os.makedirs(dst_dir, exist_ok=True)
    dst_path = os.path.join(dst_dir, fname)
    if os.path.abspath(path) == os.path.abspath(dst_path):
        return True
    shutil.move(path, dst_path)
    return True

def process_one_dir(base_dir, do_black_delete=False):
    # 대상 파일만
    exts = (".jpg", ".jpeg", ".png") if do_black_delete else None
    files = list_files_recursive(base_dir, exts=exts)

    moved = 0
    deleted = 0
    black_samples = []  # (path, mean, p99, ratio)

    t0 = time.time()

    # 1) 날짜 폴더로 이동(병렬)
    all_files = list_files_recursive(base_dir)  # 이동은 전체 파일(이미지/JSON 등)
    with ThreadPoolExecutor(max_workers=16) as ex:
        futs = [ex.submit(move_into_date_folder, p, base_dir) for p in all_files]
        for fu in tqdm(as_completed(futs), total=len(futs), desc=f"Move by date: {os.path.basename(base_dir)}"):
            if fu.result():
                moved += 1

    # 2) 블랙 삭제(이미지만, 병렬)
    if do_black_delete:
        img_files = list_files_recursive(base_dir, exts=(".jpg", ".jpeg", ".png"))
        with ThreadPoolExecutor(max_workers=16) as ex:
            futs = {ex.submit(is_black_image, p): p for p in img_files}
            for fu in tqdm(as_completed(futs), total=len(futs), desc=f"Black delete: {os.path.basename(base_dir)}"):
                p = futs[fu]
                ok, (mean, p99, ratio) = fu.result()
                if ok:
                    try:
                        os.remove(p)
                        deleted += 1
                        if len(black_samples) < 20:
                            black_samples.append((p, mean, p99, ratio))
                    except:
                        pass

    dt = time.time() - t0
    return moved, deleted, black_samples, dt

# ===== 실행 =====
# 이동 + (RGB TOP/FRONT만) 블랙 삭제
results = {}
for name, path, do_del in [
    ("RGB_TOP", DST_RGB_TOP, True),
    ("RGB_FRONT", DST_RGB_FRONT, True),
    ("THERMAL", DST_THERMAL, False),
    ("JSON", DST_JSON, False),
]:
    moved, deleted, samples, dt = process_one_dir(path, do_black_delete=do_del)
    results[name] = (moved, deleted, dt, samples)

print("\n=== Summary ===")
for k, (moved, deleted, dt, samples) in results.items():
    print(f"{k}: moved={moved:,}, deleted_black={deleted:,}, time={dt:.1f}s")
    if samples:
        print("  black examples (up to 3):")
        for p, mean, p99, ratio in samples[:3]:
            print(f"   - mean={mean:.2f}, p99={p99:.2f}, bright_ratio={ratio:.4f} | {p}")

Move by date: 260209-260223: 100%|██████████| 15083/15083 [00:39<00:00, 377.45it/s] 


=== Summary ===
RGB_TOP: moved=7,867, deleted_black=1,051, time=321.5s
  black examples (up to 3):
   - mean=1.32, p99=1.79, bright_ratio=0.0000 | /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/260209-260223/260218/bed00_20260218_120755_cam0.jpg
   - mean=1.31, p99=1.79, bright_ratio=0.0000 | /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/260209-260223/260218/bed00_20260218_131308_cam0.jpg
   - mean=1.31, p99=1.79, bright_ratio=0.0000 | /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/260209-260223/260218/bed00_20260218_155020_cam0.jpg
RGB_FRONT: moved=4,918, deleted_black=2,125, time=238.6s
  black examples (up to 3):
   - mean=2.51, p99=6.05, bright_ratio=0.0000 | /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정ᄆ

In [10]:
import os

def count_files_in_directory(path):
    count = 0
    if os.path.exists(path) and os.path.isdir(path):
        for root, dirs, files in os.walk(path):
            count += len(files)
    return count

dirs_to_count = {
    "DST_RGB_TOP": DST_RGB_TOP,
    "DST_RGB_FRONT": DST_RGB_FRONT,
    "DST_THERMAL": DST_THERMAL,
    "DST_JSON": DST_JSON
}

print("File counts in destination directories:")
for name, path in dirs_to_count.items():
    file_count = count_files_in_directory(path)
    print(f"  - {name} ({path}): {file_count:,} files")


File counts in destination directories:
  - DST_RGB_TOP (/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/260209-260223): 6,816 files
  - DST_RGB_FRONT (/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/260209-260223): 2,793 files
  - DST_THERMAL (/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/열화상/0. 원본/260209-260223): 262 files
  - DST_JSON (/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/JSON/260209-260223): 15,083 files
